Scenario: AI-Powered Study Assistant (Flashcard-Based)
1. State Definition
The assistant maintains a notebook-like state for each learner:
- topic → The subject the learner is studying (e.g., "Photosynthesis").
- flashcard → A generated Q&A card created by Groq (question on one side, answer on the other).
- progress → A log of all past flashcards attempted, including whether the learner got them correct or not.

2. Workflow (Graph of States)
Each learner interaction flows through nodes until a flashcard is delivered:
- Input Node
- Learner provides a topic or asks for practice (e.g., "Test me on cell biology").
- State updates: topic = "cell biology"
- Generation Node (Groq API)
- Groq generates a flashcard:
- flashcard.question = "What organelle is known as the powerhouse of the cell?"
- flashcard.answer = "Mitochondria"
- Response Node
- Assistant presents the flashcard question to the learner.
- Evaluation Node
- Learner responds with their answer.
- Assistant checks correctness and updates progress.
- History Node
- Logs the flashcard attempt:
- progress = [{question, learner_answer, correct/incorrect}]

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict
import requests
from google.colab import userdata

# 1. DEFINE STATE
class BotState(TypedDict):
    topic: str
    flashcard: Dict[str, str]
    learner_answer: str
    progress: List[Dict]


# 2. NODE: GENERATE FLASHCARD (Groq)
def generate_flashcard(state: BotState):
    groq_api_key = userdata.get('newapi')

    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets.")

    prompt = f"""
    Generate ONE flashcard for the topic: {state['topic']}.
    Return ONLY JSON in this format:
    {{
        "question": "...",
        "answer": "..."
    }}
    """

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    if response.status_code != 200:
        raise Exception(f"Groq API error: {response.text}")

    data = response.json()
    content = data["choices"][0]["message"]["content"]

    # Convert string JSON → dict
    try:
        flashcard = eval(content)   # simple parsing (can replace with json.loads)
    except:
        raise ValueError(f"Invalid JSON from model: {content}")

    return {"flashcard": flashcard}



# 3. NODE: ASK QUESTION
def ask_question(state: BotState):
    question = state["flashcard"]["question"]
    print(f"\nFlashcard Question:\n{question}")

    user_answer = input("\nYour Answer: ")

    return {"learner_answer": user_answer}


# 4. NODE: EVALUATE ANSWER
def evaluate_answer(state: BotState):
    correct_answer = state["flashcard"]["answer"].strip().lower()
    learner_answer = state["learner_answer"].strip().lower()

    is_correct = learner_answer == correct_answer

    print("\nCorrect Answer:", state["flashcard"]["answer"])

    if is_correct:
        print("You are CORRECT!")
    else:
        print("Incorrect.")

    return {
        "progress": state["progress"] + [{
            "question": state["flashcard"]["question"],
            "learner_answer": state["learner_answer"],
            "correct": is_correct
        }]
    }


# 5. NODE: SHOW PROGRESS
def show_progress(state: BotState):
    print("\nProgress So Far:")
    for i, entry in enumerate(state["progress"], 1):
        print(f"{i}. {entry['question']}")
        print(f"   Your Answer: {entry['learner_answer']}")
        print(f"   Result: {'✅ Correct' if entry['correct'] else '❌ Incorrect'}\n")

    return {}


# 6. BUILD GRAPH
graph = StateGraph(BotState)

graph.add_node("generate", generate_flashcard)
graph.add_node("ask", ask_question)
graph.add_node("evaluate", evaluate_answer)
graph.add_node("progress", show_progress)

# Flow
graph.set_entry_point("generate")
graph.add_edge("generate", "ask")
graph.add_edge("ask", "evaluate")
graph.add_edge("evaluate", "progress")
graph.add_edge("progress", END)

app = graph.compile()


# 7. RUN LOOP (MULTIPLE FLASHCARDS)
if __name__ == "__main__":
    topic = input("Enter topic (e.g., Biology, AI, DBMS): ")

    state = {
        "topic": topic,
        "flashcard": {},
        "learner_answer": "",
        "progress": []
    }

    while True:
        state = app.invoke(state)

        cont = input("\nDo you want another flashcard? (yes/no): ").lower()
        if cont != "yes":
            print("\nSession ended. Keep learning!")
            break

Enter topic (e.g., Biology, AI, DBMS): Biology

Flashcard Question:
What is the process by which plants convert sunlight into energy?

Your Answer: Photosynthesis

Correct Answer: Photosynthesis
You are CORRECT!

Progress So Far:
1. What is the process by which plants convert sunlight into energy?
   Your Answer: Photosynthesis
   Result: ✅ Correct


Do you want another flashcard? (yes/no): no

Session ended. Keep learning!
